# Connected RNN Perturbation Analysis

This notebook reproduces the connected RNN perturbation experiments.

The analysis compares:
- Region A neuron knockouts
- Region B neuron knockouts
- Simultaneous Region A and B knockouts

The notebook includes transition-dictionary generation/loading, perturbation experiments, Stage 2 error analysis, frozen-state analysis, behavioral-state analysis, and Stage 3 fixed-route analysis.

In [5]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
print("Working directory:", os.getcwd())

Project root: /Users/neelprabhakar/Python /Research/RNN/RNN_small_hippocampal_model
Working directory: /Users/neelprabhakar/Python /Research/RNN/RNN_small_hippocampal_model


In [6]:
import numpy as np
import torch
import matplotlib.pyplot as plt

import cm_analysis as cma
import perturbation_testing as pt
import cm_experiments as cme

print("Imports successful")

/Users/neelprabhakar/Python /.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful


## Experiment Configuration

Set the random seed and perturbation parameters used throughout the connected RNN experiment.

In [ ]:
import random

SEED = 42
TOTAL_PERTURBATIONS = 4000

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

print("Seed:", SEED)
print("Device:", device)
print("Perturbations per condition:", TOTAL_PERTURBATIONS)

## Initialize Connected RNN

Initialize the connected two-region RNN using the same random seed as the original experiment.

The model is not trained with gradient descent. Its randomly initialized weights are saved so that the original and perturbed analyses use the same model.

In [ ]:
model = cme.cm.connected_models().to(device)
model.eval()

CONNECTED_MODEL_PATH = os.path.join(
    PROJECT_ROOT,
    "post_stage1_connected_model_sd42.pt"
)

torch.save(
    model.state_dict(),
    CONNECTED_MODEL_PATH
)

print("Connected model initialized.")
print("Model saved to:", CONNECTED_MODEL_PATH)

## Original Transition Dictionaries

Generate or load the original transition dictionaries produced by the unperturbed connected RNN.

The dictionaries characterize transitions for behavioral states, each neural region independently, neural-behavioral pairs, and the joint two-region state.

In [ ]:
(
    b_transition_dict,
    all_visit_b_count_dict,
    Na_transition_dict,
    Nb_transition_dict,
    Na_Nb_transition_dict,
    Na_b_transition_dict,
    Nb_b_transition_dict,
    Na_Nb_b_transition_dict
) = cme.generate_dicts(model)

print("Original transition dictionaries ready.")
print("Behavioral states:", len(b_transition_dict))
print("Region A neural states:", len(Na_transition_dict))
print("Region B neural states:", len(Nb_transition_dict))
print("Joint neural states:", len(Na_Nb_transition_dict))
print("Joint neural-behavioral states:", len(Na_Nb_b_transition_dict))

## Connected RNN Knockout Experiments

Run three perturbation conditions:

- Region A knockout
- Region B knockout
- Simultaneous Region A and Region B knockout

At each perturbation step, a neuron is knocked out and the resulting transition behavior is compared with the original transition dictionaries.

In [ ]:
PERTURBATION_CACHE_DIR = os.path.join(
    PROJECT_ROOT,
    "Perturbation_Connected_Cache"
)

perturbation_results = cme.run_connected_knockout_experiments(
    model,
    b_transition_dict,
    Na_transition_dict,
    Nb_transition_dict,
    Na_b_transition_dict,
    Nb_b_transition_dict,
    Na_Nb_transition_dict,
    Na_Nb_b_transition_dict,
    total_perturbations=TOTAL_PERTURBATIONS,
    sd=SEED,
    save_dir=PERTURBATION_CACHE_DIR
)

print("Connected knockout experiments complete.")